In [ ]:
from geopy.distance import geodesic
from scipy.spatial import KDTree
from collections import Counter
from importlib import reload
from pyproj import Geod
import geopandas as gpd
import pandas as pd
import numpy as np
import pickle
import math
import csv
import sys
import os
sys.path.append("/u/<username>/local/pyenvs/pymeasurements")
import plot_utils as pu
import math_utils as mu
reload(pu)
reload(mu)
%matplotlib inline
import seaborn as sns
import matplotlib.pyplot as plt

## Load RTTs and flow info with geolocation

In [ ]:
ext_rtt_path    = "data/campus_trace_ext_rtt.csv"
flow_map_path   = "data/campus_trace_flow_map.csv"
geoloc_path     = "data/campus_trace_geolocation_map.csv"
ext_ip_map_path = "data/campus_trace_external_ip_map.csv"
int_ip_map_path = "data/campus_trace_internal_ip_map.csv"
conn_bytes_path = "data/conn_bytes_dict.pkl"

flow_details_output    = "data/campus_trace_flow_details.pkl"
prefix_filtered_output = "data/campus_trace_prefixes_filtered_on_minrtt.pkl"
rtts_with_prefixes_output = "data/campus_trace_rtts_with_prefixes.pkl"
prefixes_rtts_geo_path = "data/campus_traces_prefixes_rtts_geo.pkl"
rtts_with_prefixes_geo_usa_path = "data/campus_traces_rtts_with_prefixes_geo_usa.pkl"

### Load geolocation data

In [ ]:
# Geolocation map
df_geo = pd.read_csv(geoloc_path, low_memory=False)

# Convert latitude and longitude to numeric, coercing errors to NaN
df_geo['Latitude']  = pd.to_numeric(df_geo['Latitude'], errors='coerce')
df_geo['Longitude'] = pd.to_numeric(df_geo['Longitude'], errors='coerce')

# Drop rows where latitude or longitude is NaN
df_geo = df_geo.dropna(subset=['Latitude', 'Longitude'])

print("df_geo row count:", df_geo.shape[0])

In [ ]:
df_geo.head()

### Load flow data

In [ ]:
# Flows
## Note: Source IP/port is always the internal IP/port and destination IP/port is always the external IP/port
df_flows_without_geo = pd.read_csv(flow_map_path)
print("df_flows_without_geo row count:", df_flows_without_geo.shape[0])

In [ ]:
df_flows_without_geo.head()

### Deanonymize

In [ ]:
ext_ip_map = {}
with open(ext_ip_map_path) as fp:
    for line in [l.strip() for l in fp.readlines()][1:]:
        tokens = line.split(",")
        ext_ip_map[tokens[0]] = tokens[1]
print(f"No. of external IPs: {len(ext_ip_map)}")

In [ ]:
int_ip_map = {}
with open(int_ip_map_path) as fp:
    for line in [l.strip() for l in fp.readlines()][1:]:
        tokens = line.split(",")
        int_ip_map[tokens[0]] = tokens[1]
print(f"No. of internal IPs: {len(int_ip_map)}")

In [ ]:
def get_trace_source_ip(row):
    if row['Source_IP'] in int_ip_map:
        return int_ip_map[row['Source_IP']]
    else:
        print(f"Source IP not found in map: {row['Source_IP']}")

def get_trace_destination_ip(row):
    if row['Destination_IP'] in ext_ip_map:
        return ext_ip_map[row['Destination_IP']]
    else:
        print("Destination IP not found in map: {row[]}")

In [ ]:
df_flows_without_geo_deanon = df_flows_without_geo.copy()
df_flows_without_geo_deanon['Source_IP']      = df_flows_without_geo_deanon.apply(get_trace_source_ip, axis=1)
df_flows_without_geo_deanon['Destination_IP'] = df_flows_without_geo_deanon.apply(get_trace_destination_ip, axis=1)
df_flows_without_geo_deanon.head()

In [ ]:
def count_unique_prefixes(ips):
    prefixes = set()
    for ip in ips:
        prefixes.add('.'.join(ip.split('.')[:3]) + '.0')
    return len(prefixes)

print(f"Before geolocation:: No. of unique flows: {df_flows_without_geo_deanon['Flow_ID'].nunique()}")
print(f"Before geolocation:: No. of unique external IPs: {df_flows_without_geo_deanon['Destination_IP'].nunique()}")
print(f"Before geolocation:: No. of unique external prefixes: {count_unique_prefixes(df_flows_without_geo_deanon['Destination_IP'].tolist())}")

### Enrich with connection bytes and host type information

In [ ]:
conn_bytes = None
with open("data/conn_bytes_dict.pkl", "rb") as fp:
    conn_bytes = pickle.load(fp)

In [ ]:
def internal_host_type(row):
    if row['Source_Port'] > 1023 and row['Destination_Port'] < 1024:
        return 'CISO' # Client in PU
    if row['Source_Port'] < 1024 and row['Destination_Port'] > 1023:
        return 'SICO' # Server in PU
    if row['Source_Port'] > 1023 and row['Destination_Port'] > 1023:
        return 'CICO'
    if row['Source_Port'] < 1024 and row['Destination_Port'] < 1023:
        return 'SISO'
    return 'Neither'

def get_conn_bytes(row):
    conn_id = (row['Source_IP'], row['Source_Port'], row['Destination_IP'], row['Destination_Port'])
    return conn_bytes[conn_id]

In [ ]:
df_flows_without_geo_deanon_enriched = df_flows_without_geo_deanon.copy()
df_flows_without_geo_deanon_enriched['Connection_Type']  = df_flows_without_geo_deanon_enriched.apply(internal_host_type, axis=1)
df_flows_without_geo_deanon_enriched['Connection_Bytes'] = df_flows_without_geo_deanon_enriched.apply(get_conn_bytes, axis=1)
df_flows_without_geo_deanon_enriched.head()

### Add geolocation information

In [ ]:
df_flows_detailed = pd.merge(df_flows_without_geo_deanon_enriched, df_geo, on='Destination_IP', how='inner')
print(f"df_flows_detailed row count: {df_flows_detailed.shape[0]}" \
       + f" ({round(df_flows_detailed.shape[0]*100/df_flows_without_geo_deanon_enriched.shape[0], 2)}%)")

In [ ]:
df_flows_detailed.head(n=1)

In [ ]:
print(f"After geolocation:: No. of unique flows: {df_flows_detailed['Flow_ID'].nunique()}")
print(f"After geolocation:: No. of unique external IPs: {df_flows_detailed['Destination_IP'].nunique()}")
print(f"After geolocation:: No. of unique external prefixes: {df_flows_detailed['Destination_Prefix'].nunique()}")

In [ ]:
# Retain only needed columns
df_flows = df_flows_detailed[ [
    'Flow_ID', 'Destination_Prefix', 'Destination_Prefix_Anon', 'Connection_Type', 'Connection_Bytes',
    'Continent', 'Country', 'Latitude', 'Longitude'] ]
df_flows.head(n=1)

## RTTs

### Load RTT information

In [ ]:
# RTTs
df_rtts = pd.read_csv(ext_rtt_path)
print("df_rtts row count:", df_rtts.shape[0])

In [ ]:
df_rtts.head(n=1)

### Per-flow minRTT

In [ ]:
df_rtts_minrtt = df_rtts.groupby('Flow_ID').agg(
    Start_Timestamp=('ACK_Timestamp', 'min'),
    End_Timestamp=('ACK_Timestamp', 'max'),
    RTT_Count=('RTT_ms', 'count'),
    minRTT_ms=('RTT_ms', 'min')
).reset_index()
df_rtts_minrtt['Duration_s'] = df_rtts_minrtt['End_Timestamp'] - df_rtts_minrtt['Start_Timestamp']

# Rearrange columns
df_rtts_minrtt = df_rtts_minrtt[ ['Flow_ID', 'Start_Timestamp', 'End_Timestamp', 'Duration_s', 'minRTT_ms', 'RTT_Count'] ]

print("df_rtts_minrtt row count:", df_rtts_minrtt.shape[0])

In [ ]:
df_rtts_minrtt.head(n=1)

### Enrich flow data with RTT data

In [ ]:
df_flows_minrtt = pd.merge(df_flows, df_rtts_minrtt, on='Flow_ID', how='inner')

# Reorder the columns
df_flows_minrtt = df_flows_minrtt[ [
    'Flow_ID', 'Destination_Prefix', 'Destination_Prefix_Anon', 'Connection_Type', 'Connection_Bytes',
    'Start_Timestamp', 'End_Timestamp', 'Duration_s', 'RTT_Count', 'minRTT_ms',
    'Continent', 'Country', 'Latitude', 'Longitude'
] ]

print("df_flows_minrtt row count:", df_flows_minrtt.shape[0])

In [ ]:
df_flows_minrtt.head(n=1)

In [ ]:
df_flows_minrtt.to_pickle(flow_details_output)

## Aggregation by prefix

In [ ]:
df_prefixes_minrtt_nonagg = df_flows_minrtt.groupby('Destination_Prefix').agg({
    'Destination_Prefix_Anon': list,
    'Flow_ID': list,
    'Connection_Type': list,
    'Connection_Bytes': list,
    'Start_Timestamp': list,
    'End_Timestamp': list,
    'Duration_s': list,
    'RTT_Count': list,
    'minRTT_ms': list,
    'Continent': list,
    'Country': list,
    'Latitude': list,
    'Longitude': list
}).reset_index()
print(f"Shape of df_prefixes_minrtt_nonagg: {df_prefixes_minrtt_nonagg.shape[0]}")

In [ ]:
df_prefixes_minrtt_nonagg.head(n=1)

In [ ]:
def identity(x):
    return x

def add_elements(x):
    return sum(x)

def unique_elements(x):
    return list(set(x))

def count_elements(x):
    return len(x)

def count_unique_elements(x):
    return len(list(set(x)))

def min_elements(x):
    return min(x)

def max_elements(x):
    return max(x)

In [ ]:
# Define aggregations for each column
aggregations = {
    'Destination_Prefix': identity,
    'Destination_Prefix_Anon': unique_elements,
    'Flow_ID': count_unique_elements,
    'Connection_Type': unique_elements,
    'Connection_Bytes': add_elements,
    'Start_Timestamp': min_elements,
    'End_Timestamp': max_elements,
    'Duration_s': add_elements,
    'RTT_Count': add_elements,
    'minRTT_ms': min_elements,
    'Continent': unique_elements,
    'Country': unique_elements,
    'Latitude': unique_elements,
    'Longitude': unique_elements
}

df_prefixes_minrtt_agg = pd.DataFrame({
    col: df_prefixes_minrtt_nonagg[col].apply(aggregations[col]) for col in list(aggregations.keys())
})

df_prefixes_minrtt_agg['Prefix_Duration_s'] = df_prefixes_minrtt_agg['End_Timestamp'] - df_prefixes_minrtt_agg['Start_Timestamp']

df_prefixes_minrtt_agg = df_prefixes_minrtt_agg.rename(columns={
    'Flow_ID': 'Flow_Count',
    'Connection_Type': 'Connection_Type_Unique',
    'Connection_Bytes': 'Prefix_Total_Bytes',
    'Duration_s': 'Flows_Total_Duration_s',
    'Continent': 'Continents_Unique',
    'Country': 'Country_Unique',
    'Latitude': 'Latitude_Unique',
    'Longitude': 'Longitude_Unique'
})

print(f"Shape of df_prefixes_minrtt_agg: {df_prefixes_minrtt_agg.shape[0]}"
     + f" ({round(df_prefixes_minrtt_agg.shape[0]*100/df_prefixes_minrtt_nonagg.shape[0], 2)}%)")

In [ ]:
df_prefixes_minrtt_agg.head(n=1)

In [ ]:
df_prefixes_minrtt_agg_filtered = df_prefixes_minrtt_agg[
    (df_prefixes_minrtt_agg['Destination_Prefix_Anon'].apply(lambda x: len(x) == 1))
    & (df_prefixes_minrtt_agg['Continents_Unique'].apply(lambda x: len(x) == 1))
    & (df_prefixes_minrtt_agg['Country_Unique'].apply(lambda x: len(x) == 1))
    & (df_prefixes_minrtt_agg['Latitude_Unique'].apply(lambda x: len(x) == 1))
    & (df_prefixes_minrtt_agg['Longitude_Unique'].apply(lambda x: len(x) == 1))
]
print(f"Shape of df_prefixes_minrtt_agg_filtered: {df_prefixes_minrtt_agg_filtered.shape[0]}" \
      + f" ({round(df_prefixes_minrtt_agg_filtered.shape[0]*100/df_prefixes_minrtt_agg.shape[0], 2)}%)")

In [ ]:
df_prefixes_minrtt_agg_filtered.head(n=1)

In [ ]:
def get_distance_from_princeton(row):
    src_coord = (40.343899, -74.660049)
    dst_coord = (float(row['Latitude']), float(row['Longitude']))
    distance = geodesic(src_coord, dst_coord).km
    return distance

In [ ]:
def compute_rtt_lb(row):
    dist_m = row['Geodesic_Distance_km'] * 1000
    c = 299792458
    rtt = (2*dist_m)/(2*c/3)
    return rtt * 1000

In [ ]:
df_prefixes_minrtt = df_prefixes_minrtt_agg_filtered.copy()
df_prefixes_minrtt['Destination_Prefix_Anon'] = df_prefixes_minrtt['Destination_Prefix_Anon'].apply(lambda x: x[0])
df_prefixes_minrtt['Continent'] = df_prefixes_minrtt['Continents_Unique'].apply(lambda x: x[0])
df_prefixes_minrtt['Country'] = df_prefixes_minrtt['Country_Unique'].apply(lambda x: x[0])
df_prefixes_minrtt['Latitude'] = df_prefixes_minrtt['Latitude_Unique'].apply(lambda x: x[0])
df_prefixes_minrtt['Longitude'] = df_prefixes_minrtt['Longitude_Unique'].apply(lambda x: x[0])
df_prefixes_minrtt['Geodesic_Distance_km'] = df_prefixes_minrtt.apply(get_distance_from_princeton, axis=1)

df_prefixes_minrtt = df_prefixes_minrtt[[
    'Destination_Prefix', 'Destination_Prefix_Anon', 'Flow_Count', 'Connection_Type_Unique', 'Prefix_Total_Bytes',
    'Start_Timestamp', 'End_Timestamp', 'Prefix_Duration_s', 'Flows_Total_Duration_s', 'RTT_Count', 'minRTT_ms', 
    'Continent', 'Country', 'Latitude', 'Longitude', 'Geodesic_Distance_km'
]]

df_prefixes_minrtt['minRTT_Lower_Bound_ms'] = df_prefixes_minrtt.apply(compute_rtt_lb, axis=1)

print(f"Shape of df_prefixes_minrtt: {df_prefixes_minrtt.shape[0]}")

In [ ]:
df_prefixes_minrtt.head(n=1)

In [ ]:
df_prefixes_minrtt_filtered = df_prefixes_minrtt[
    (df_prefixes_minrtt['minRTT_ms'] >= df_prefixes_minrtt['minRTT_Lower_Bound_ms'])
]

print(f"Shape of df_prefixes_minrtt_filtered: {df_prefixes_minrtt_filtered.shape[0]}"
      + f" ({mu.rnd(df_prefixes_minrtt_filtered.shape[0]*100/df_prefixes_minrtt.shape[0], 2)}%)")
print(f"Total flow count: {df_prefixes_minrtt_filtered['Flow_Count'].sum()}"
      + f" ({mu.rnd(df_prefixes_minrtt_filtered['Flow_Count'].sum()*100/df_prefixes_minrtt['Flow_Count'].sum(), 2)}%)")
print(f"Total RTT count: {df_prefixes_minrtt_filtered['RTT_Count'].sum()}"
      + f" ({mu.rnd(df_prefixes_minrtt_filtered['RTT_Count'].sum()*100/df_prefixes_minrtt['RTT_Count'].sum(), 2)}%)")
print(f"No. of countries: {df_prefixes_minrtt_filtered['Country'].nunique()}"
      + f" ({mu.rnd(df_prefixes_minrtt_filtered['Country'].nunique()*100/df_prefixes_minrtt['Country'].nunique(), 2)}%)")

In [ ]:
df_prefixes_minrtt_filtered.head(n=1)

In [ ]:
df_prefixes_minrtt_filtered.to_pickle(prefix_filtered_output)

### Breakdown of prefixes by connection types

In [ ]:
# Prefix types with a single connection type: CISO, SICO, CICO, SISO

df_prefixes_ciso = df_prefixes_minrtt_filtered[
    df_prefixes_minrtt_filtered['Connection_Type_Unique'].apply(
        lambda x: len(x) == 1 and x[0] == 'CISO')]

df_prefixes_sico = df_prefixes_minrtt_filtered[
    df_prefixes_minrtt_filtered['Connection_Type_Unique'].apply(
        lambda x: len(x) == 1 and x[0] == 'SICO')]

df_prefixes_cico = df_prefixes_minrtt_filtered[
    df_prefixes_minrtt_filtered['Connection_Type_Unique'].apply(
        lambda x: len(x) == 1 and x[0] == 'CICO')]

df_prefixes_siso = df_prefixes_minrtt_filtered[
    df_prefixes_minrtt_filtered['Connection_Type_Unique'].apply(
        lambda x: len(x) == 1 and x[0] == 'SISO')]

prefix_1type_count = df_prefixes_ciso.shape[0] + df_prefixes_sico.shape[0] + df_prefixes_cico.shape[0] + df_prefixes_siso.shape[0]
print(f"Prefix types with a single connection type: {prefix_1type_count}" \
     + f" ({round(prefix_1type_count*100/df_prefixes_minrtt_filtered.shape[0], 2)}%)")
print(f"Shape of df_prefixes_ciso: {df_prefixes_ciso.shape[0]}" \
      + f" ({round(df_prefixes_ciso.shape[0]*100/df_prefixes_minrtt_filtered.shape[0], 2)}%)")
print(f"Shape of df_prefixes_sico: {df_prefixes_sico.shape[0]}" \
      + f" ({round(df_prefixes_sico.shape[0]*100/df_prefixes_minrtt_filtered.shape[0], 2)}%)")
print(f"Shape of df_prefixes_cico: {df_prefixes_cico.shape[0]}" \
      + f" ({round(df_prefixes_cico.shape[0]*100/df_prefixes_minrtt_filtered.shape[0], 2)}%)")
print(f"Shape of df_prefixes_siso: {df_prefixes_siso.shape[0]}" \
      + f" ({round(df_prefixes_siso.shape[0]*100/df_prefixes_minrtt_filtered.shape[0], 2)}%)")

In [ ]:
# Mixed prefix types with 2 connection types: {CISO, SICO}, {CISO, CICO}, {CISO, SISO}, {SICO, CICO}, {SICO, SISO}, {CICO, SISO}

df_prefixes_ciso_sico = df_prefixes_minrtt_filtered[
    df_prefixes_minrtt_filtered['Connection_Type_Unique'].apply(
        lambda x: len(x) == 2 and 'CISO' in x and 'SICO' in x)]

df_prefixes_ciso_cico = df_prefixes_minrtt_filtered[
    df_prefixes_minrtt_filtered['Connection_Type_Unique'].apply(
        lambda x: len(x) == 2 and 'CISO' in x and 'CICO' in x)]

df_prefixes_ciso_siso = df_prefixes_minrtt_filtered[
    df_prefixes_minrtt_filtered['Connection_Type_Unique'].apply(
        lambda x: len(x) == 2 and 'CISO' in x and 'SISO' in x)]

df_prefixes_sico_cico = df_prefixes_minrtt_filtered[
    df_prefixes_minrtt_filtered['Connection_Type_Unique'].apply(
        lambda x: len(x) == 2 and 'SICO' in x and 'CICO' in x)]

df_prefixes_sico_siso = df_prefixes_minrtt_filtered[
    df_prefixes_minrtt_filtered['Connection_Type_Unique'].apply(
        lambda x: len(x) == 2 and 'SICO' in x and 'SISO' in x)]

df_prefixes_cico_siso = df_prefixes_minrtt_filtered[
    df_prefixes_minrtt_filtered['Connection_Type_Unique'].apply(
        lambda x: len(x) == 2 and 'CICO' in x and 'SISO' in x)]

prefix_2type_count = df_prefixes_ciso_sico.shape[0] + df_prefixes_ciso_cico.shape[0] + df_prefixes_ciso_siso.shape[0] \
                        + df_prefixes_sico_cico.shape[0] + df_prefixes_sico_siso.shape[0] + df_prefixes_cico_siso.shape[0]
print(f"Prefix types with a double connection type: {prefix_2type_count}" \
     + f" ({round(prefix_2type_count*100/df_prefixes_minrtt_filtered.shape[0], 2)}%)")
print(f"Shape of df_prefixes_ciso_cico: {df_prefixes_ciso_cico.shape[0]}" \
      + f" ({round(df_prefixes_ciso_cico.shape[0]*100/df_prefixes_minrtt_filtered.shape[0], 2)}%)")
print(f"Shape of df_prefixes_ciso_sico: {df_prefixes_ciso_sico.shape[0]}" \
      + f" ({round(df_prefixes_ciso_sico.shape[0]*100/df_prefixes_minrtt_filtered.shape[0], 2)}%)")
print(f"Shape of df_prefixes_ciso_siso: {df_prefixes_ciso_siso.shape[0]}" \
      + f" ({round(df_prefixes_ciso_siso.shape[0]*100/df_prefixes_minrtt_filtered.shape[0], 2)}%)")
print(f"Shape of df_prefixes_sico_cico: {df_prefixes_sico_cico.shape[0]}" \
      + f" ({round(df_prefixes_sico_cico.shape[0]*100/df_prefixes_minrtt_filtered.shape[0], 2)}%)")
print(f"Shape of df_prefixes_sico_siso: {df_prefixes_sico_siso.shape[0]}" \
      + f" ({round(df_prefixes_sico_siso.shape[0]*100/df_prefixes_minrtt_filtered.shape[0], 2)}%)")
print(f"Shape of df_prefixes_cico_siso: {df_prefixes_cico_siso.shape[0]}" \
      + f" ({round(df_prefixes_cico_siso.shape[0]*100/df_prefixes_minrtt_filtered.shape[0], 2)}%)")

In [ ]:
# Mixed prefix types with 3 connection types: {CISO, SICO, CICO}, {CISO, SICO, SISO}, {CISO, CICO, SISO}, {SICO, CICO, SISO}

df_prefixes_ciso_sico_cico = df_prefixes_minrtt_filtered[
    df_prefixes_minrtt_filtered['Connection_Type_Unique'].apply(
        lambda x: len(x) == 3 and 'CISO' in x and 'SICO' in x and 'CICO' in x)]

df_prefixes_ciso_sico_siso = df_prefixes_minrtt_filtered[
    df_prefixes_minrtt_filtered['Connection_Type_Unique'].apply(
        lambda x: len(x) == 3 and 'CISO' in x and 'SICO' in x and 'SISO' in x)]

df_prefixes_ciso_cico_siso = df_prefixes_minrtt_filtered[
    df_prefixes_minrtt_filtered['Connection_Type_Unique'].apply(
        lambda x: len(x) == 3 and 'CISO' in x and 'CICO' in x and 'SISO' in x)]

df_prefixes_sico_cico_siso = df_prefixes_minrtt_filtered[
    df_prefixes_minrtt_filtered['Connection_Type_Unique'].apply(
        lambda x: len(x) == 3 and 'SICO' in x and 'CICO' in x and 'SISO' in x)]

prefix_3type_count = df_prefixes_ciso_sico_cico.shape[0] + df_prefixes_ciso_sico_siso.shape[0] \
                        + df_prefixes_ciso_cico_siso.shape[0] + df_prefixes_sico_cico_siso.shape[0]
print(f"Prefix types with a triple connection type: {prefix_3type_count}" \
     + f" ({round(prefix_3type_count*100/df_prefixes_minrtt_filtered.shape[0], 2)}%)")
print(f"Shape of df_prefixes_ciso_sico_cico: {df_prefixes_ciso_sico_cico.shape[0]}" \
      + f" ({round(df_prefixes_ciso_sico_cico.shape[0]*100/df_prefixes_minrtt_filtered.shape[0], 2)}%)")
print(f"Shape of df_prefixes_ciso_sico_siso: {df_prefixes_ciso_sico_siso.shape[0]}" \
      + f" ({round(df_prefixes_ciso_sico_siso.shape[0]*100/df_prefixes_minrtt_filtered.shape[0], 2)}%)")
print(f"Shape of df_prefixes_ciso_cico_siso: {df_prefixes_ciso_cico_siso.shape[0]}" \
      + f" ({round(df_prefixes_ciso_cico_siso.shape[0]*100/df_prefixes_minrtt_filtered.shape[0], 2)}%)")
print(f"Shape of df_prefixes_sico_cico_siso: {df_prefixes_sico_cico_siso.shape[0]}" \
      + f" ({round(df_prefixes_sico_cico_siso.shape[0]*100/df_prefixes_minrtt_filtered.shape[0], 2)}%)")

### Breakdown of flows by connection types

In [ ]:
print(f"Total no. of flows: {df_prefixes_minrtt_filtered['Flow_Count'].sum()}")
print(f"No. of CISO flows: {df_prefixes_ciso['Flow_Count'].sum()}" \
      + f" ({round(df_prefixes_ciso['Flow_Count'].sum()*100/df_prefixes_minrtt_filtered['Flow_Count'].sum(), 2)}%)")
print(f"No. of SICO flows: {df_prefixes_sico['Flow_Count'].sum()}" \
      + f" ({round(df_prefixes_sico['Flow_Count'].sum()*100/df_prefixes_minrtt_filtered['Flow_Count'].sum(), 2)}%)")
print(f"No. of CICO flows: {df_prefixes_cico['Flow_Count'].sum()}" \
      + f" ({round(df_prefixes_cico['Flow_Count'].sum()*100/df_prefixes_minrtt_filtered['Flow_Count'].sum(), 2)}%)")
print(f"No. of SISO flows: {df_prefixes_siso['Flow_Count'].sum()}" \
      + f" ({round(df_prefixes_siso['Flow_Count'].sum()*100/df_prefixes_minrtt_filtered['Flow_Count'].sum(), 2)}%)")

## RTTs enriched with prefix identifiers

In [ ]:
df_flows_minrtt_prefix = df_flows_minrtt[['Flow_ID', 'Destination_Prefix', 'Destination_Prefix_Anon']]
df_flows_minrtt_prefix.head(n=1)

In [ ]:
df_rtts_with_prefixes = pd.merge(df_flows_minrtt_prefix, df_rtts, on='Flow_ID', how='left')
df_rtts_with_prefixes.head(n=1)

In [ ]:
print(f"No. of RTT samples: {df_rtts_with_prefixes.shape[0]}")

In [ ]:
df_rtts_with_prefixes.to_pickle(rtts_with_prefixes_output)

## Prefixes and geolocations and RTTs

In [ ]:
df_prefixes_with_rtts = df_rtts_with_prefixes[['Destination_Prefix', 'Flow_ID', 'ACK_Timestamp', 'RTT_ms']].copy()
df_prefixes_with_rtts = df_prefixes_with_rtts.sort_values(['Destination_Prefix', 'ACK_Timestamp'])
df_prefixes_with_rtts = df_prefixes_with_rtts.groupby('Destination_Prefix').agg({
    'Flow_ID': list,
    'ACK_Timestamp': list,
    'RTT_ms': list
}).reset_index()
df_prefixes_with_rtts.head(n=1)

In [ ]:
df_prefixes_minrtt_filtered_select = df_prefixes_minrtt_filtered.drop(columns=['Destination_Prefix_Anon', 'Continent'])
df_prefixes_with_rtts_geo = pd.merge(df_prefixes_minrtt_filtered_select, df_prefixes_with_rtts, on='Destination_Prefix', how='inner')
df_prefixes_with_rtts_geo.head(n=1)

In [ ]:
df_prefixes_with_rtts_geo.to_pickle(prefixes_rtts_geo_path)

### Anecdotes

In [ ]:
anecdote_prefix_row = df_prefixes_with_rtts_geo[df_prefixes_with_rtts_geo["Destination_Prefix"] == "148.59.195.0"]
anecdote_prefix_fids = anecdote_prefix_row.Flow_ID.iloc[0]
anecdote_prefix_ts   = anecdote_prefix_row.ACK_Timestamp.iloc[0]
anecdote_prefix_rtts = anecdote_prefix_row.RTT_ms.iloc[0]

time = {0: [], 1: []}
rtts = {0: [], 1: []}

ts_offset = min(df_prefixes_with_rtts_geo.Start_Timestamp.tolist())

for f, t_abs, r in zip(anecdote_prefix_fids, anecdote_prefix_ts, anecdote_prefix_rtts):
    t = t_abs - ts_offset
    if t >= ts_start and t <= ts_end:
        if f == "f7389191":
            time[0].append(t)
            rtts[0].append(r)
        elif f == "f7414116":
            time[1].append(t)
            rtts[1].append(r)

ts_start = max(min(time[0]), min(time[1]))
ts_end   = min(max(time[0]), max(time[1]))

time_temp = time.copy()
rtts_temp = rtts.copy()
time = {0: [], 1: []}
rtts = {0: [], 1: []}

for i in [0, 1]:
    for t, r in zip(time_temp[i], rtts_temp[i]):
        if t - ts_start >= 0 and t - ts_start <= 25 and r <= 300:
            time[i].append(t - ts_start)
            rtts[i].append(r)

reload(pu)
sns_colors = sns.color_palette("bright")
pu.scatterplots([time[0], time[1]], [rtts[0], rtts[1]],
               {
                   "curvelabels": ["Flow 1", "Flow 2"],
                    "ylim": (90, 310), "markersize": 75, "alpha": 0.5,
                    "colors": [sns_colors[0], sns_colors[3]],
                    "markers": ["o"], "loc": "upper right", "framealpha": 1.0
               })

In [ ]:
start_time = 0
end_time = 25
window_size = 0.25

window_ends = []
window_minrtts_0 = []
window_minrtts_1 = []
window_prefix_minrtts = []

for t in np.arange(start_time, end_time, window_size):
    # Window covers [t, t + 0.25)
    window_start = t
    if window_start == 0:
        window_start = -0.1
    window_end = t + 0.25
    
    # Boolean mask for data in this window
    mask_0 = (np.array(time[0]) > window_start) & (np.array(time[0]) <= window_end)
    rtts_in_window_0 = np.array(rtts[0])[mask_0]
    mask_1 = (np.array(time[1]) > window_start) & (np.array(time[1]) <= window_end)
    rtts_in_window_1 = np.array(rtts[1])[mask_1]
    
    if len(rtts_in_window_0) > 0:
        # Compute the minimum in this window
        min_rtt_0 = rtts_in_window_0.min()

    if len(rtts_in_window_1) > 0:
        # Compute the minimum in this window
        min_rtt_1 = rtts_in_window_1.min()
    
    # Store the end of the window (or midpoint—whatever you prefer)
    window_ends.append(window_end)
    window_minrtts_0.append(min_rtt_0)
    window_minrtts_1.append(min_rtt_1)
    window_prefix_minrtts.append(min(min_rtt_0, min_rtt_1))

reload(pu)

plt.rc('xtick',labelsize=23)
plt.rc('ytick',labelsize=23)
plt.rc('axes',labelsize=25)

fig, (ax_top, ax_bottom) = plt.subplots(
    nrows=2, 
    figsize=(8, 8), 
    sharex=True  # share x-axis so times line up
)

colors = list(sns.color_palette("bright"))

# Top subplot: All RTTs
ax_top.scatter(time[0], rtts[0], color=colors[0], s=75, marker="o", alpha=0.9, label="Flow 1")
ax_top.scatter(time[1], rtts[1], color=colors[3], s=75, marker="o", alpha=0.9, label="Flow 2")
ax_top.legend(loc="upper right", framealpha=1)
ax_top.set_ylabel('RTT (ms)')
ax_top.set_xlim(-0.5, 25.5)
ax_top.set_ylim(91, 308)

# Bottom subplot: Windowed Minimum RTTs
ax_bottom.scatter(window_ends, window_minrtts_0, color=colors[0], s=75, marker="x", alpha=1, label="Flow 1")
ax_bottom.scatter(window_ends, window_minrtts_1, color=colors[3], s=75, marker="x", alpha=1, label="Flow 2")
ax_bottom.scatter(window_ends, window_prefix_minrtts, color=colors[2], marker='^', s=75, alpha=0.6, label="Prefix")
ax_bottom.legend(loc="upper right", framealpha=1)
ax_bottom.set_xlabel('Time (s)')
ax_bottom.set_ylabel('Min. RTT (ms)')
ax_bottom.set_ylim(91, 308)

plt.tight_layout()
plt.savefig("plots/prefix_aggregation.pdf", dpi=300)
plt.show()

### Bitcoin RTTs

In [ ]:
df_bitcoin_rtts = pd.read_csv("data/bitcoin_attack.csv", sep=" ")
df_bitcoin_acknums = pd.read_csv("data/bitcoin_attack_acknums.csv")
df_bitcoin_acknums_unique = df_bitcoin_acknums.drop_duplicates(subset='tcp.ack').rename(
    columns = {"tcp.ack": "ack_num", "frame.time_relative": "time"})
df_bitcoin_rtts = pd.merge(df_bitcoin_rtts, df_bitcoin_acknums_unique, on='ack_num', how='inner')

bitcoin_time = df_bitcoin_rtts["time"].tolist()
bitcoin_rtts = df_bitcoin_rtts["rtt_ms"].tolist()
bitcoin_rtts = [r-20 for r in bitcoin_rtts]

offset = 34

bitcoin_time = [t-offset for t in bitcoin_time][offset+42:]
bitcoin_rtts = bitcoin_rtts[offset+42:]

bitcoin_time_temp = bitcoin_time.copy()
bitcoin_rtts_temp = bitcoin_rtts.copy()

bitcoin_time = []
bitcoin_rtts = []

for t, r in zip(bitcoin_time_temp, bitcoin_rtts_temp):
    if t >= 50 and t <= 250:
        bitcoin_time.append(t-50)
        bitcoin_rtts.append(r)

pu.scatterplot(bitcoin_time, bitcoin_rtts,
               {
                    "xlabel": "Time (s)", "ylabel": "RTT (ms)", "title": "",
                    "ylim": (90, 150), "markers": ["o"], "markersize": 75,
                    "plot_path": "plots/bitcoin_attack_methodology.pdf"
               })

In [ ]:
start_time = 0
end_time = 200
window_size = 1.5

window_ends = []
window_minrtts = []

for t in np.arange(start_time, end_time, window_size):
    # Window covers [t, t + 0.25)
    window_start = t
    if window_start == 0:
        window_start = -0.1
    window_end = t + 1
    
    # Boolean mask for data in this window
    mask = (np.array(bitcoin_time) > window_start) & (np.array(bitcoin_time) <= window_end)
    rtts_in_window = np.array(bitcoin_rtts)[mask]
    
    if len(rtts_in_window) > 0:
        # Compute the minimum in this window
        min_rtt = rtts_in_window.min()
    
    # Store the end of the window (or midpoint—whatever you prefer)
    window_ends.append(window_end)
    window_minrtts.append(min_rtt)

plt.rc('xtick',labelsize=23)
plt.rc('ytick',labelsize=23)
plt.rc('axes',labelsize=25)

fig, (ax_top, ax_bottom) = plt.subplots(
    nrows=2, 
    figsize=(8, 8), 
    sharex=True  # share x-axis so times line up
)

colors = list(sns.color_palette("bright"))

# Top subplot: All RTTs
ax_top.scatter(bitcoin_time, bitcoin_rtts, color=colors[0], s=75, marker="o")
ax_top.set_ylabel('RTT (ms)')
ax_top.set_xlim(-10, 210)
ax_top.set_ylim(90, 150)

# Bottom subplot: Windowed Minimum RTTs
ax_bottom.scatter(window_ends, window_minrtts, color=colors[2], s=75, marker="^", alpha=1)
ax_bottom.set_xlabel('Time (s)')
ax_bottom.set_ylabel('Min. RTT (ms)')
ax_bottom.set_ylim(90, 150)

plt.tight_layout()
plt.savefig("plots/prefix_aggregation.pdf", dpi=300)
plt.show()